<a href="https://colab.research.google.com/github/cras-lab/ML-examples/blob/main/KNN_OpenAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

OpenAI를 설치한다.

In [ ]:
pip install -q openai

외부에서 API키를 입력 받는다.

In [ ]:
import getpass
api_key = getpass.getpass("Enter your OPENAI_API_KEY: ")

필요한 모듈을 임포트 한다.

In [10]:
import numpy as np
from openai import OpenAI
from sklearn.neighbors import NearestNeighbors

OpenAI 객체를 생성한다.

In [11]:
client = OpenAI(api_key=api_key)

학생 피드백 예제를 설정한다.

In [12]:
sentences = [
    "강의 속도가 너무 빨라서 따라가기 어려웠습니다.",
    "조금 천천히 설명해 주시면 좋겠습니다.",
    "설명이 전반적으로 명확하고 이해하기 쉬웠습니다.",
    "예제가 많아서 개념을 이해하는 데 큰 도움이 됐습니다.",
    "실습 예제가 더 많았으면 좋겠습니다.",
    "과제가 생각보다 어려웠지만 많이 배웠습니다.",
    "과제 난이도가 너무 높아서 부담이 컸습니다.",
    "실습 시간이 부족해서 아쉬웠습니다.",
    "코드 시연이 있어서 좋았습니다.",
    "이론 설명이 체계적이라 좋았습니다.",
    "시험 범위를 조금 더 구체적으로 알려주시면 좋겠습니다.",
    "평가 기준이 명확했으면 좋겠습니다.",
    "실제 사례를 많이 보여줘서 흥미로웠습니다.",
    "수업 자료가 잘 정리되어 있어서 복습하기 좋았습니다.",
    "과제 피드백을 더 자세히 받고 싶습니다."
]

OpenAI의 임베딩 객체를 생성하고, 임베딩을 수행한다.

In [13]:
embed_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=sentences
)

임베딩을 배열에 저장한다.

In [14]:
doc_embeddings = np.array(
    [item.embedding for item in embed_response.data],
    dtype=np.float32
)

코사인 유사도를 통해 KNN을 수행한다.

In [ ]:
knn = NearestNeighbors(n_neighbors=3, metric="cosine")
knn.fit(doc_embeddings)

루프를 돌며 질문에 대답한다.

In [ ]:
print("\n질문을 입력하세요. 종료하려면 exit 입력.\n")

# 질문 루프
while True:
    question = input("질문: ").strip()

    if question.lower() in ["exit", "quit", "종료"]:
        print("실습을 종료합니다.")
        break

    if not question:
        print("질문이 비어 있습니다. 다시 입력하세요.\n")
        continue

    # 6. 질문 임베딩
    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[question]
    )

    query_embedding = np.array(
        query_response.data[0].embedding,
        dtype=np.float32
    ).reshape(1, -1)

    # 7. 유사 문장 검색
    distances, indices = knn.kneighbors(query_embedding)
    retrieved = [sentences[i] for i in indices[0]]

    print("\n[검색된 문장]")
    for s in retrieved:
        print("-", s)

    # 8. 근거 기반 답변 생성
    prompt = f"""
당신은 수업 피드백 분석 도우미다.
반드시 아래 피드백만 근거로 사용해서 한국어로 2문장 이내로 답하라.
근거에 없는 내용은 추가하지 마라.

질문: {question}

피드백:
- {retrieved[0]}
- {retrieved[1]}
- {retrieved[2]}
"""

    response = client.responses.create(
        model="gpt-5-mini",
        input=prompt
    )

    print("\n[OpenAI 답변]")
    print(response.output_text)
    print("\n" + "=" * 50 + "\n")